In [1]:
import os
import time
import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
import StringAPI as string
import gseapy as gp
from pathlib import Path
from io import StringIO
from matplotlib.lines import Line2D


# Illustrator settings
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


# Paths
main_dir = '/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309'
#main_dir = 'C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309'

ref_dir = Path(main_dir, "_set_data")

out_dir = Path(main_dir, "STRING_AGM_analysisnewnames")
out_dir.mkdir(exist_ok=True)


# Read gene list data
#df = pd.read_csv(Path(main_dir, "_set_data/PLSsig_genelists_long.csv"))
df = pd.read_csv(Path(main_dir, "STRING_AGM_analysis/Annotations/PLSgenelist_long_HGNCcorrected.csv"), index_col=False)


# STRING channel settings
channels = {
    "neighborhood": False,
    "fusion": False,
    "cooccurrence": False,
    "coexpression": True,
    "experiments": True,
    "database": False,
    "textmining": True
}


# Enrichr libraries
libraries = [
    "KEGG_2021_Human",
    "GO_Biological_Process_2023"
]


In [2]:
df['gene']=df['gene_renamed']

In [3]:
# ----------------------------
# Loop through AGMs
# ----------------------------
for module_name in df["AGM_ID"].unique():

    print("\n" + "=" * 80)
    print(f"Running AGM: {module_name}")
    print("=" * 80)

    # Get cell type and reference genes
    celltype = module_name.split("_")[0]

    ref_genes = pd.read_csv(
        Path(ref_dir, f"{celltype}_ref_genes.txt"),
        header=None
    )[0]

    ref_genes = (
        ref_genes
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    # Get genes for this AGM
    genes = (df.loc[df["AGM_ID"] == module_name, "gene"].astype(str).tolist())
    print(f"Number of genes: {len(genes)}")
    print(f"Number of background genes: {len(ref_genes)}")

    # Get STRING network and connected groups
    network_df = string.query_string_network_with_channels(genes,channels=channels)

    groups_df = string.get_string_connected_groups(network_df,genes,include_singletons=False)

    groups_df.to_csv(
        Path(out_dir, f"{module_name}_STRING_groups.csv"),
        index=False)

    # Save STRING network image
    string.save_string_network_images(
        genes=genes,
        out_prefix=out_dir / f"{module_name}_STRING_network",
        channels=channels,
        hide_disconnected_nodes=True,
        save_svg=True,
        save_png=True,
        save_highres_png=False
    )

    # Annotate STRING groups
    gr_desc_df = string.annotate_string_connected_groups(
        groups_df,
        module_name
    )

    gr_desc_df.to_csv(
        out_dir / f"{module_name}_STRING_groups_descriptions.csv",
        index=False
    )

    # Subset unannotated groups
    unannotated_groups = gr_desc_df.loc[
        gr_desc_df["primary_description"] == "-",
        ["STRING_group", "group_size", "genes"]
    ]

    # ----------------------------
    # Run Enrichr for unannotated groups
    # ----------------------------
    unanngr_enrichr_df = string.run_enrichr_for_unannotated_groups(
        unannotated_groups=unannotated_groups,
        module_name=module_name,
        ref_genes=ref_genes,
        libraries=libraries)

    unanngr_enrichr_df.to_csv(
        out_dir / f"{module_name}_unannotated_groups_enrichr.csv",
        index=False
    )

    print(f"Finished AGM: {module_name}")


Running AGM: Astrocytes_AstDPP10_Cluster_6
Number of genes: 87
Number of background genes: 2213
Saved STRING svg to: /Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309/STRING_AGM_analysisnewnames/Astrocytes_AstDPP10_Cluster_6_STRING_network.svg
Saved STRING image to: /Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309/STRING_AGM_analysisnewnames/Astrocytes_AstDPP10_Cluster_6_STRING_network.png
Running Astrocytes_AstDPP10_Cluster_6 | Group_1 | 4 group genes | 2213 background genes
Running Astrocytes_AstDPP10_Cluster_6 | Group_2 | 4 group genes | 2213 background genes
Running Astrocytes_AstDPP10_Cluster_6 | Group_3 | 4 group genes | 2213 background genes
Running Astrocytes_AstDPP10_Cluster_6 | Group_4 | 2 group genes | 2213 background genes
Skipping Group_4: fewer than 3 genes
Running Astrocytes_AstDPP10_Cluster_6 | Group_5 | 2 group genes | 2213 background genes
Skipping Group_5: fewer than 3 genes
Running Astrocytes_AstDPP10_Cluster_6 | Group_6 | 2 g

/opt/anaconda3/envs/SEAADpy/lib/python3.12/site-packages/gseapy/enrichr.py:264: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


Running Oligodendrocytes_Oli_Cluster_3 | Group_5 | 2 group genes | 2391 background genes
Skipping Group_5: fewer than 3 genes
Running Oligodendrocytes_Oli_Cluster_3 | Group_6 | 2 group genes | 2391 background genes
Skipping Group_6: fewer than 3 genes
Running Oligodendrocytes_Oli_Cluster_3 | Group_7 | 2 group genes | 2391 background genes
Skipping Group_7: fewer than 3 genes
Finished AGM: Oligodendrocytes_Oli_Cluster_3
